# Step 4: LLM Inference

Kilian Lüders & Hannah Birkenkötter

Using ChatGPT to extract information on the actor in selected paragraphs.

**Steps:**
1. Load Data
2. Set Up LLM
3. Inference
4. Save Results

**Input:** `data/subset_demand_request_decide.pkl` (from `3_data_analysis.ipynb`) (1235 x 22) Subset of `data/data_res.pkl`.

**Output:** `data/subset_demand_request_decide_llm.pkl`

In [ ]:
import json
from tqdm import tqdm
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = input()

### 1. Load Data

In [ ]:
data = pd.read_pickle("data/subset_demand_request_decide.pkl")
data['sent'] = data.text.apply(lambda x: "The Security Council " + x)
data[['doc', 'text', 'sent']].head()
print(data.shape)
data.head()

In [ ]:
data.doc.nunique()

### 2. Set Up LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Your task is to analyze the given clause from the Security Council and answer a question.
The clause contains at least one of the following verbs: 'demand', 'request', 'authorize', or 'decide'.
Which actor is addressed in the clause with the verbs 'request', 'demand', 'authorize', 'decide'? Note that a clause may address multiple actors.
List all relevant actors in your response.

Your output should be a JSON object containing the following information:

{{
    "actors": List // List of actors based on the clause
}}
"""),
    ("human", "{clause}")
])

In [ ]:
prompt

In [ ]:
chain = prompt | llm

In [ ]:
data_text = data['sent'].to_list()

### 3. Inference

In [ ]:
results = []


for i, txt in enumerate(tqdm(data_text), start=1):
    try:
        result = chain.invoke({"clause": txt})
        results.append(result)
    except Exception as e:
        print("Exception Occurred:", e)
        results.append("")

    if i % 100 == 0:
        pd.DataFrame({'data': results}).to_pickle("data/tmp.pkl")
pd.DataFrame({'data': results}).to_pickle("data/tmp.pkl")

### 4. Save Results

In [ ]:
categories_list = [json.loads(x.content)["actors"] for x in results]
data['actors'] = categories_list
data.to_pickle("data/subset_demand_request_decide_llm.pkl")
data.head()